In [ ]:
import gradio as gr
import requests
import os

api_key = "sk-你的真实密钥填在这里"
url = "https://api.deepseek.com/chat/completions"
file_path = "chat_history.txt"  # 记录保存的文件名

# 1. 定义一个函数：把网页上的聊天记录存到 txt 里
def save_history(history):
    # 'w' 表示覆盖写入，把最新的全部历史重写一遍
    with open(file_path, "w", encoding="utf-8") as f:
        for user_msg, ai_msg in history:
            f.write(f"你: {user_msg}\n")
            f.write(f"AI: {ai_msg}\n")
            f.write("-" * 20 + "\n")

# 2. 定义核心逻辑：读取历史文件 + 请求大模型
def chat_with_ai(user_message, history):
    if not user_message:
        return "", history

    # 构建发给大模型的消息列表
    messages = []
    
    # 如果你希望每次打开网页时，AI 都记得以前的聊天，先把 txt 读出来
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            # 这里只是为了简单展示，真实产品中需要更复杂的解析
            # 我们默认让大模型大概知道之前聊过就行，这里略过把 txt 转格式的复杂步骤
            pass

    # 转换 Gradio 的网页历史记录给大模型
    for item in history:
        messages.append({"role": "user", "content": item[0]})
        messages.append({"role": "assistant", "content": item[1]})
    messages.append({"role": "user", "content": user_message})
    
    # 请求 API
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "deepseek-chat",
        "messages": messages
    }
    
    try:
        response = requests.post(url, headers=headers, json=data)
        result = response.json()
        ai_reply = result["choices"][0]["message"]["content"]
        
        # 把这一轮的新对话加入历史记录，并保存到 txt
        history.append((user_message, ai_reply))
        save_history(history)
        
        # 返回空字符串清空输入框，返回更新后的 history 渲染到网页
        return "", history
    except Exception as e:
        return f"出错啦: {e}", history

# 3. 搭建网页
with gr.Blocks(title="有记忆的AI") as demo:
    gr.Markdown("# 💾 你的 AI 记忆存储版")
    gr.Markdown(f"当前所有聊天记录都会自动保存到电脑上的 `{file_path}` 文件中。")
    
    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="说点什么吧...")
    
    # 提交事件：输入内容，传历史，输出清空输入框和更新历史
    msg.submit(chat_with_ai, [msg, chatbot], [msg, chatbot])

demo.launch(share=True)